# Network Intrusion Detection

## Historical Context (Narrative)
The KDD Cup 1999 dataset defined a generation of IDS research. It contained 41 features from simulated network connections, with attack categories: DoS, Probe, R2L, U2R. Most 2000s-era IDS papers optimized accuracy on KDD99, achieving >99% — creating false confidence. KDD99 is heavily preprocessed, temporally leaky, and not representative of real traffic. The first paper to seriously challenge this was NSL-KDD (2009).

Modern IDS datasets: **CICIDS2017** (Canadian Institute for Cybersecurity), **UNSW-NB15**, **CIC-DDoS2019**. These capture real packet captures converted to flow features using CICFlowMeter.

## Modern Approach
Current network IDS uses GBM (XGBoost/LightGBM) for flow-based detection, achieving practical performance. Autoencoders add anomaly detection for zero-day detection. The key engineering challenge is feature engineering from raw flows.

In [ ]:
import numpy as np
from typing import Dict, List, Tuple

np.random.seed(42)

# Simulate CICIDS2017-style network flow features
# Real CICIDS2017 has ~80 features; we simulate key ones

FEATURE_NAMES = [
    'duration', 'protocol_type', 'fwd_packets', 'bwd_packets',
    'fwd_bytes', 'bwd_bytes', 'flow_bytes_per_sec', 'flow_packets_per_sec',
    'fwd_iat_mean', 'bwd_iat_mean', 'psh_flag_count', 'syn_flag_count',
    'ack_flag_count', 'fin_flag_count', 'rst_flag_count',
    'fwd_header_length', 'bwd_header_length', 'down_up_ratio',
    'avg_fwd_segment', 'avg_bwd_segment',
]

def generate_normal_flow(n: int) -> np.ndarray:
    """Simulate normal web/DNS traffic flows."""
    duration = np.random.exponential(2.0, n)  # seconds
    protocol = np.random.choice([6, 17], n)   # TCP=6, UDP=17
    fwd_pkts = np.random.poisson(15, n)
    bwd_pkts = np.random.poisson(12, n)
    fwd_bytes = fwd_pkts * np.random.randint(100, 1500, n)
    bwd_bytes = bwd_pkts * np.random.randint(100, 1500, n)
    flow_bps = (fwd_bytes + bwd_bytes) / (duration + 0.001)
    flow_pps = (fwd_pkts + bwd_pkts) / (duration + 0.001)
    fwd_iat = np.random.exponential(0.1, n)
    bwd_iat = np.random.exponential(0.1, n)
    psh = np.random.poisson(2, n)
    syn = np.ones(n)  # one SYN per connection
    ack = fwd_pkts
    fin = np.ones(n)
    rst = np.zeros(n)
    fwd_hdr = np.ones(n) * 20
    bwd_hdr = np.ones(n) * 20
    ratio = bwd_bytes / (fwd_bytes + 1)
    avg_fwd = fwd_bytes / (fwd_pkts + 1)
    avg_bwd = bwd_bytes / (bwd_pkts + 1)
    return np.column_stack([
        duration, protocol, fwd_pkts, bwd_pkts, fwd_bytes, bwd_bytes,
        flow_bps, flow_pps, fwd_iat, bwd_iat, psh, syn, ack, fin, rst,
        fwd_hdr, bwd_hdr, ratio, avg_fwd, avg_bwd
    ])

def generate_dos_flow(n: int) -> np.ndarray:
    """Simulate DoS attack flows: high rate, small packets, minimal response."""
    duration = np.random.exponential(0.2, n)  # short bursts
    protocol = np.ones(n) * 6  # TCP
    fwd_pkts = np.random.poisson(500, n)  # flood
    bwd_pkts = np.random.poisson(2, n)    # victim barely responds
    fwd_bytes = fwd_pkts * np.random.randint(40, 80, n)  # small SYN packets
    bwd_bytes = bwd_pkts * np.random.randint(40, 60, n)
    flow_bps = (fwd_bytes + bwd_bytes) / (duration + 0.001)
    flow_pps = (fwd_pkts + bwd_pkts) / (duration + 0.001)
    fwd_iat = np.random.exponential(0.0002, n)  # very fast
    bwd_iat = np.random.exponential(1.0, n)
    psh = np.zeros(n)
    syn = np.random.poisson(100, n)  # SYN flood
    ack = np.zeros(n)
    fin = np.zeros(n)
    rst = np.random.poisson(5, n)
    fwd_hdr = np.ones(n) * 20
    bwd_hdr = np.ones(n) * 20
    ratio = bwd_bytes / (fwd_bytes + 1)
    avg_fwd = fwd_bytes / (fwd_pkts + 1)
    avg_bwd = bwd_bytes / (bwd_pkts + 1)
    return np.column_stack([
        duration, protocol, fwd_pkts, bwd_pkts, fwd_bytes, bwd_bytes,
        flow_bps, flow_pps, fwd_iat, bwd_iat, psh, syn, ack, fin, rst,
        fwd_hdr, bwd_hdr, ratio, avg_fwd, avg_bwd
    ])

# Generate dataset
X_normal = generate_normal_flow(800)
X_dos = generate_dos_flow(200)
X = np.vstack([X_normal, X_dos])
y = np.array([0]*800 + [1]*200)

print(f'Dataset: {len(X)} flows, {X.shape[1]} features')
print(f'Normal: {(y==0).sum()}, DoS: {(y==1).sum()}')


In [ ]:
# GBM-style classifier using feature importance
# (Simplified decision tree ensemble)

class SimpleDecisionStump:
    def __init__(self):
        self.feature = None
        self.threshold = None
        self.left_val = None
        self.right_val = None
    
    def fit(self, X, residuals, sample_weight=None):
        n, d = X.shape
        best_loss = np.inf
        
        for feat in range(d):
            thresholds = np.percentile(X[:, feat], [25, 50, 75])
            for thresh in thresholds:
                left_mask = X[:, feat] <= thresh
                right_mask = ~left_mask
                if left_mask.sum() < 5 or right_mask.sum() < 5:
                    continue
                lv = residuals[left_mask].mean()
                rv = residuals[right_mask].mean()
                loss = ((residuals[left_mask] - lv)**2).sum() + ((residuals[right_mask] - rv)**2).sum()
                if loss < best_loss:
                    best_loss = loss
                    self.feature = feat
                    self.threshold = thresh
                    self.left_val = lv
                    self.right_val = rv
        return self
    
    def predict(self, X):
        mask = X[:, self.feature] <= self.threshold
        out = np.where(mask, self.left_val, self.right_val)
        return out

class GradientBoostingClassifier:
    def __init__(self, n_estimators=50, lr=0.1):
        self.n_estimators = n_estimators
        self.lr = lr
        self.stumps = []
        self.init_pred = 0.0
    
    def sigmoid(self, z): return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
    
    def fit(self, X, y):
        self.init_pred = np.log(y.mean() / (1 - y.mean() + 1e-8))
        F = np.full(len(y), self.init_pred)
        
        for _ in range(self.n_estimators):
            p = self.sigmoid(F)
            residuals = y - p  # gradient of log-loss
            stump = SimpleDecisionStump()
            stump.fit(X, residuals)
            self.stumps.append(stump)
            F += self.lr * stump.predict(X)
        return self
    
    def predict_proba(self, X):
        F = np.full(len(X), self.init_pred)
        for stump in self.stumps:
            F += self.lr * stump.predict(X)
        return self.sigmoid(F)
    
    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) > threshold).astype(int)

# Normalize and split
perm = np.random.permutation(len(X))
X_s, y_s = X[perm], y[perm]
split = 800
X_tr, y_tr = X_s[:split], y_s[:split]
X_te, y_te = X_s[split:], y_s[split:]

mu, sigma = X_tr.mean(0), X_tr.std(0) + 1
X_tr_n = (X_tr - mu) / sigma
X_te_n = (X_te - mu) / sigma

gbc = GradientBoostingClassifier(n_estimators=30, lr=0.2)
gbc.fit(X_tr_n, y_tr)

preds = gbc.predict(X_te_n)
acc = (preds == y_te).mean()
tp = ((preds == 1) & (y_te == 1)).sum()
fp = ((preds == 1) & (y_te == 0)).sum()
fn = ((preds == 0) & (y_te == 1)).sum()
precision = tp / (tp + fp) if tp + fp > 0 else 0
recall = tp / (tp + fn) if tp + fn > 0 else 0

print(f'GBM IDS Results:')
print(f'  Accuracy:  {acc:.3f}')
print(f'  Precision: {precision:.3f}')
print(f'  Recall:    {recall:.3f}')
print(f'  F1:        {2*precision*recall/(precision+recall+1e-8):.3f}')


In [ ]:
# Autoencoder-based anomaly detection for zero-day detection
# Train only on normal traffic; high reconstruction error = anomaly

class NetworkAutoencoder:
    """Simple autoencoder for network flow anomaly detection."""
    
    def __init__(self, input_dim: int, bottleneck_dim: int = 8, lr: float = 0.01):
        # Encoder: input -> bottleneck
        self.W1 = np.random.randn(input_dim, bottleneck_dim) * 0.1
        self.b1 = np.zeros(bottleneck_dim)
        # Decoder: bottleneck -> output
        self.W2 = np.random.randn(bottleneck_dim, input_dim) * 0.1
        self.b2 = np.zeros(input_dim)
        self.lr = lr
    
    def relu(self, x): return np.maximum(0, x)
    def relu_grad(self, x): return (x > 0).astype(float)
    
    def forward(self, X):
        h = self.relu(X @ self.W1 + self.b1)
        out = h @ self.W2 + self.b2  # linear output layer
        return out, h
    
    def fit(self, X_normal, epochs=100, batch_size=64):
        n = len(X_normal)
        for ep in range(epochs):
            idx = np.random.permutation(n)
            for start in range(0, n, batch_size):
                batch = X_normal[idx[start:start+batch_size]]
                out, h = self.forward(batch)
                err = out - batch
                dW2 = h.T @ err / len(batch)
                db2 = err.mean(0)
                dh = err @ self.W2.T * self.relu_grad(batch @ self.W1 + self.b1)
                dW1 = batch.T @ dh / len(batch)
                db1 = dh.mean(0)
                self.W2 -= self.lr * dW2
                self.b2 -= self.lr * db2
                self.W1 -= self.lr * dW1
                self.b1 -= self.lr * db1
        return self
    
    def reconstruction_error(self, X):
        out, _ = self.forward(X)
        return np.mean((X - out)**2, axis=1)

# Train autoencoder ONLY on normal traffic
X_normal_n = (X_normal - mu) / sigma
X_dos_n = (X_dos - mu) / sigma

ae = NetworkAutoencoder(input_dim=20, bottleneck_dim=6)
ae.fit(X_normal_n, epochs=50)

# Reconstruction errors
normal_errs = ae.reconstruction_error(X_normal_n[:100])
dos_errs = ae.reconstruction_error(X_dos_n[:100])

threshold = np.percentile(normal_errs, 95)
dos_detected = (dos_errs > threshold).mean()

print(f'Autoencoder Anomaly Detector:')
print(f'  Normal recon error: mean={normal_errs.mean():.4f}, 95th={threshold:.4f}')
print(f'  DoS recon error:    mean={dos_errs.mean():.4f}')
print(f'  DoS detection rate at 5% FPR threshold: {dos_detected:.3f}')
